# Eval & Observability: Introduction to Evaluation

Welcome to Module 06! Now that we can build sophisticated agentic systems with Reflection, Plan-and-Execute, and multi-agent teams, a critical question emerges: **How do we know if our agent is actually good?**

Traditional software testing (unit tests, integration tests) doesn't fully apply to LLM-based systems because outputs are non-deterministic. The same prompt can yield different phrasing, structure, or reasoning paths every time.

This module introduces **Evaluation Strategies** for agentic AI:
1. **Heuristic Evaluation**: Rule-based checks (e.g., "Does the output contain a code block?", "Is the response under 500 words?").
2. **LLM-as-a-Judge**: Using a separate, powerful LLM to grade the output of your agent against a rubric.
3. **Reference-Based Evaluation**: Comparing agent output to a known-good "golden" answer.

---

## 1. Environment Setup
We'll use LangChain and Google Gemini for our evaluations.

In [1]:
import os
import json
from typing import List
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()
# We use temperature=0 for the judge to ensure consistent grading
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
# The "agent" LLM we are evaluating — higher temperature for creative outputs
agent_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

print("Evaluation Environment Ready!")

/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:234: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with cri

Evaluation Environment Ready!


## 2. Heuristic Evaluation
The simplest form of evaluation: write Python functions that check structural properties of the output. No LLM needed — just deterministic rules.

This is fast, cheap, and reliable for checking format compliance.

In [2]:
def eval_contains_code_block(response: str) -> bool:
    """Check if the response contains a markdown code block."""
    return "```" in response

def eval_max_length(response: str, max_words: int = 500) -> bool:
    """Check if the response is within the word limit."""
    return len(response.split()) <= max_words

def eval_no_apology(response: str) -> bool:
    """Check that the agent does not apologize (a common LLM anti-pattern)."""
    apology_phrases = ["i'm sorry", "i apologize", "sorry about that", "my apologies"]
    return not any(phrase in response.lower() for phrase in apology_phrases)

# --- Test the heuristic evaluators ---
test_query = "Write a Python function to reverse a string."
response = agent_llm.invoke([HumanMessage(content=test_query)])
agent_output = response.content

print("--- Agent Output ---")
print(agent_output[:300], "...")
print("\n--- Heuristic Evaluation Results ---")
print(f"  Contains Code Block: {eval_contains_code_block(agent_output)}")
print(f"  Within 500 Words:    {eval_max_length(agent_output)}")
print(f"  No Apology:          {eval_no_apology(agent_output)}")

--- Agent Output ---
You can reverse a string in Python using several methods. Here are a few common approaches, each with its own advantages:

---

### Method 1: Using Slicing (The Most Pythonic Way)

This is generally considered the most concise and Pythonic way to reverse a string.

```python
def reverse_string_slice ...

--- Heuristic Evaluation Results ---
  Contains Code Block: True
  Within 500 Words:    False
  No Apology:          True


## 3. LLM-as-a-Judge
Heuristics can check format, but they can't check **quality**. Is the code correct? Is the explanation clear? For this, we use a powerful LLM as a grader.

The key insight: LLMs are generally better at *evaluating* than *generating*. A model that might write buggy code can often reliably *spot* bugs in code.

In [3]:
# Pydantic schema for structured judge output
class EvalResult(BaseModel):
    """Structured evaluation result from the LLM judge."""
    score: int = Field(description="Score from 1-5. 1=Terrible, 3=Acceptable, 5=Excellent.")
    reasoning: str = Field(description="Detailed justification for the score.")
    has_bugs: bool = Field(description="True if the code contains logical bugs or errors.")

judge_llm_structured = judge_llm.with_structured_output(EvalResult)

In [4]:
def llm_judge_evaluate(query: str, agent_response: str) -> EvalResult:
    """Use an LLM to grade the agent's response."""
    system_msg = SystemMessage(content="""You are a strict, expert code reviewer.
    You are evaluating an AI agent's response to a user's coding request.
    
    Grade the response on:
    1. Correctness — Does the code solve the problem?
    2. Completeness — Are edge cases handled?
    3. Code Quality — Docstrings, type hints, readability?
    4. Explanation — Is the accompanying text clear and helpful?
    
    Be very strict. A score of 5 means production-ready code.""")
    
    eval_prompt = f"""ORIGINAL USER REQUEST:
{query}

AGENT'S RESPONSE:
{agent_response}

Evaluate the response above."""
    
    result = judge_llm_structured.invoke([system_msg, HumanMessage(content=eval_prompt)])
    return result

# --- Run the LLM-as-a-Judge ---
eval_result = llm_judge_evaluate(test_query, agent_output)

print("--- LLM-as-a-Judge Evaluation ---")
print(f"  Score:    {eval_result.score}/5")
print(f"  Has Bugs: {eval_result.has_bugs}")
print(f"  Reasoning: {eval_result.reasoning}")

--- LLM-as-a-Judge Evaluation ---
  Score:    5/5
  Has Bugs: False
  Reasoning: The agent's response is outstanding. It provides four distinct, correct, and well-explained methods for reversing a string, including the most Pythonic approach. Each function includes excellent docstrings, type hints, and clear example usage that covers edge cases like empty strings and single-character strings. The explanations for each method are precise and helpful, with a valuable note on efficiency for the loop-based method. Furthermore, the 'Which method to choose?' section offers excellent guidance, correctly identifying the preferred methods. This response goes above and beyond the user's request, demonstrating a deep understanding of Python best practices and pedagogical clarity. The code is production-ready in terms of correctness, completeness, and quality.


## 4. Reference-Based Evaluation
When you have a known-good "golden" answer, you can ask the judge to compare the agent's output against it. This is useful for regression testing — ensuring that model upgrades don't degrade quality.

In [5]:
class ReferenceEvalResult(BaseModel):
    """Evaluation comparing agent output to a golden reference."""
    score: int = Field(description="Score from 1-5. 5 means equivalent or better than the reference.")
    matches_reference: bool = Field(description="True if the agent's output is functionally equivalent.")
    differences: str = Field(description="Key differences from the reference answer.")

ref_judge = judge_llm.with_structured_output(ReferenceEvalResult)

# Our known-good "golden" answer
golden_answer = """def reverse_string(s: str) -> str:
    \"\"\"Reverse a string.
    
    Args:
        s: The input string.
    
    Returns:
        The reversed string.
    \"\"\"
    return s[::-1]"""

def reference_evaluate(query: str, agent_response: str, reference: str) -> ReferenceEvalResult:
    """Compare agent output to a golden reference answer."""
    system_msg = SystemMessage(content="""You are comparing an AI agent's response to a known-correct reference answer.
    Focus on FUNCTIONAL equivalence, not exact text matching.
    The agent's approach may differ but still be correct.""")
    
    prompt = f"""USER REQUEST: {query}

REFERENCE ANSWER:
{reference}

AGENT'S ANSWER:
{agent_response}

Compare the agent's answer to the reference."""
    
    return ref_judge.invoke([system_msg, HumanMessage(content=prompt)])

ref_result = reference_evaluate(test_query, agent_output, golden_answer)

print("--- Reference-Based Evaluation ---")
print(f"  Score:            {ref_result.score}/5")
print(f"  Matches Reference: {ref_result.matches_reference}")
print(f"  Differences:       {ref_result.differences}")

--- Reference-Based Evaluation ---
  Score:            5/5
  Matches Reference: True
  Differences:       The agent provides multiple correct methods for reversing a string, including the one used in the reference, along with detailed explanations and examples. It is more comprehensive and educational than the reference.


## 5. Batch Evaluation — Building a Test Suite
A single test proves nothing. Production evaluation requires running your agent against a **dataset** of test cases and aggregating the scores.

In [7]:
# Define a small test suite
test_suite = [
    {"query": "Write a Python function to check if a number is prime.", "difficulty": "easy"},
    {"query": "Write a Python function that merges two sorted lists into one sorted list.", "difficulty": "medium"},
    {"query": "Write a Python class that implements a thread-safe Singleton pattern.", "difficulty": "hard"},
]

results = []
for i, test_case in enumerate(test_suite):
    print(f"\n--- Test Case {i+1}: [{test_case['difficulty'].upper()}] ---")
    print(f"Query: {test_case['query']}")
    
    # 1. Generate response
    response = agent_llm.invoke([HumanMessage(content=test_case["query"])])
    
    # 2. Evaluate with heuristics
    has_code = eval_contains_code_block(response.content)
    within_limit = eval_max_length(response.content)
    no_apology = eval_no_apology(response.content)
    
    # 3. Evaluate with LLM judge
    judge_result = llm_judge_evaluate(test_case["query"], response.content)
    
    results.append({
        "query": test_case["query"],
        "difficulty": test_case["difficulty"],
        "heuristic_code": has_code,
        "heuristic_length": within_limit,
        "heuristic_no_apology": no_apology,
        "judge_score": judge_result.score,
        "judge_has_bugs": judge_result.has_bugs,
    })
    
    print(f"  Heuristic Pass: Code={has_code}, Length={within_limit}, NoApology={no_apology}")
    print(f"  Judge Score: {judge_result.score}/5 | Bugs: {judge_result.has_bugs}")

# --- Aggregate Results ---
avg_score = sum(r["judge_score"] for r in results) / len(results)
bug_rate = sum(1 for r in results if r["judge_has_bugs"]) / len(results) * 100
heuristic_pass_rate = sum(1 for r in results if r["heuristic_code"] and r["heuristic_length"] and r["heuristic_no_apology"]) / len(results) * 100

print("\n" + "="*50)
print(f"AGGREGATE RESULTS ({len(results)} test cases)")
print(f"  Average Judge Score:  {avg_score:.1f}/5")
print(f"  Bug Rate:             {bug_rate:.0f}%")
print(f"  Heuristic Pass Rate:  {heuristic_pass_rate:.0f}%")
print("="*50)


--- Test Case 1: [EASY] ---
Query: Write a Python function to check if a number is prime.
  Heuristic Pass: Code=True, Length=False, NoApology=True
  Judge Score: 4/5 | Bugs: False

--- Test Case 2: [MEDIUM] ---
Query: Write a Python function that merges two sorted lists into one sorted list.
  Heuristic Pass: Code=True, Length=False, NoApology=True
  Judge Score: 5/5 | Bugs: False

--- Test Case 3: [HARD] ---
Query: Write a Python class that implements a thread-safe Singleton pattern.
  Heuristic Pass: Code=True, Length=False, NoApology=True
  Judge Score: 4/5 | Bugs: False

AGGREGATE RESULTS (3 test cases)
  Average Judge Score:  4.3/5
  Bug Rate:             0%
  Heuristic Pass Rate:  0%


## Summary

Evaluation is the **backbone** of production-grade agentic AI.

| Strategy | Speed | Cost | Best For |
| :--- | :--- | :--- | :--- |
| **Heuristic** | ⚡ Instant | Free | Format checks, length limits, anti-pattern detection |
| **LLM-as-a-Judge** | 🐢 Slow | $$ | Quality, correctness, reasoning depth |
| **Reference-Based** | 🐢 Slow | $$ | Regression testing, model comparison |

> **Pro-Tip**: In production, always combine heuristic and LLM-based evaluation. Run heuristics first (they're free) to catch obvious failures, then use the LLM judge only on responses that pass the heuristic gate. This dramatically reduces API costs.

In the next notebook, we'll explore how to **observe** what happens *inside* the agent using LangSmith tracing.